# Milestone 2: conventional CNN grounding baseline

A separate 145,329-parameter CNN, trained from scratch on the same four arrangements.
**8,640 updates / 40 passes / 276,480 QA presentations**, one CUDA device, float32.
Compare final transfer to the four-arrangement transformer (44.34% aggregate).
Equal updates and exposure do not imply equal compute. No recurrence comparison.

Enable GPU and internet; push the completed source archive to the repository root.
Set a committed `REPO_REF` before running. No reserved validation/test inference.
See `docs/milestones/milestone2_cnn_baseline.md` for the review and fixed protocol.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Krailon/multi-modal-loop-llm.git"
REPO_REF = "milestone2"  # Commit SHA preferred; must include this implementation.
REPO_DIR = Path("/kaggle/working/multi-modal-loop-cnn")
RUN_ROOT = Path("/kaggle/working/milestone2_cnn_baseline")
SOURCE = REPO_DIR / "milestone2_multi_arrangement_artifacts.zip"

## Checkout and install

Run cells in order. Keep Kaggle’s installed PyTorch.


In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR).resolve()
RUN_ROOT = Path(RUN_ROOT).resolve()
if REPO_DIR == RUN_ROOT or REPO_DIR.is_relative_to(RUN_ROOT) or RUN_ROOT.is_relative_to(REPO_DIR):
    raise ValueError("Repository and artifacts must use separate directories")
if not REPO_REF or REPO_REF.startswith("-"):
    raise ValueError("Set REPO_REF to a branch, tag, or commit")


def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()


if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
else:
    if git("remote", "get-url", "origin") != REPO_URL:
        raise ValueError("Existing checkout belongs to a different repository")
    if git("status", "--porcelain"):
        raise ValueError("Existing checkout has local changes; use a clean committed checkout")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    if git("rev-parse", "HEAD") != git("rev-parse", "FETCH_HEAD"):
        raise ValueError(
            "Existing checkout has a different revision. Set REPO_REF to its recorded commit "
            "or use a new REPO_DIR for a new run."
        )

resolved_revision = git("rev-parse", "HEAD")
identity = (str(REPO_DIR), resolved_revision)
if globals().get("_notebook_code_identity", identity) != identity:
    raise RuntimeError(
        "The kernel previously loaded another revision; restart it before proceeding"
    )
_notebook_code_identity = identity
print("Resolved code revision:", resolved_revision)
torch_version = importlib.metadata.version("torch")
# No torch extra, requirements.txt, or accelerator replacement.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)
if importlib.metadata.version("torch") != torch_version:
    raise RuntimeError("PyTorch changed during installation; inspect the environment")
# Editable-install .pth files are processed at interpreter startup. Make this
# checkout importable immediately in the running notebook kernel too.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
# Training/evaluation run in subprocesses using this kernel's Python interpreter.

## Audit and prepare

Pinned source predictions and manifest are checked without loading reference weights.
Fresh seed-0 training uses learned/02/04/06; final transfer uses 01/03/05/07.
All quartets, pixels, questions and answers are preserved.


In [ ]:
import os

import torch

from multimodal_loop.eval.kaggle_cnn_baseline import (
    archive_cnn_baseline,
    prepare_cnn_baseline,
    run_cnn_baseline,
)

os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
torch.set_num_threads(2)
if not SOURCE.is_file():
    raise FileNotFoundError(f"Push the completed archive to the repo root: {SOURCE}")
run = prepare_cnn_baseline(REPO_DIR, RUN_ROOT, SOURCE)
print("Manifest:", run.manifest_sha256)
print("Budget: 8,640 updates; 40 presentations per QA. Training fit monitored only.")

## Train and assess the final checkpoint

No resume, early stopping, extension or best-checkpoint selection. A failed fit is
inconclusive. Strong transfer would motivate a separate visual-stem experiment;
it would not identify which transformer component caused the gap.


In [ ]:
report = run_cnn_baseline(run)

## Read the results

First check training fit, then **every transfer arrangement**. Improvement concentrated
in one row does not establish broad transfer. Families require both circle/square
answers correct across all four size combinations. Transfer has no new pass/fail gate.


In [ ]:
import json

from IPython.display import Markdown, display

comparison = json.loads((RUN_ROOT / "diagnosis/comparison.json").read_text())
print("Training criteria:", "PASS" if report["assessment"]["passed"] else "NOT MET")
for role in ("training_fit", "transfer"):
    metrics = report["aggregates"][role]
    print(
        f"{role}: {metrics['summary']['accuracy']:.2%} accuracy; "
        f"{metrics['families']['fraction']:.2%} correct families"
    )
rows = [
    "| Transfer arrangement | Transformer | CNN | Change | Correct families |",
    "| --- | ---: | ---: | ---: | ---: |",
]
for name, result in comparison["arrangements"].items():
    accuracy = result["accuracy"]
    families = report["arrangements"][name]["families"]
    rows.append(
        f"| {name} | {accuracy['reference']:.2%} | {accuracy['current']:.2%} | "
        f"{100 * accuracy['difference']:+.2f} pp | "
        f"{families['fraction']:.2%} of {families['total']} |"
    )
display(Markdown("\n".join(rows)))
print("Runtime:", json.loads((RUN_ROOT / "training/runtime.json").read_text()))
print("Selected errors:", RUN_ROOT / "diagnosis/inspection.html")
print("Single-seed exploratory control. No milestone completion or recurrence claim.")

## Download artifacts

Return this archive to the local repository root for review.


In [ ]:
from IPython.display import FileLink, display

archive = archive_cnn_baseline(run)
print(archive)
display(FileLink(str(archive)))